In [1]:
!git clone https://github.com/Radhika-Amar-Desai/BSF_implementation.git

Cloning into 'BSF_implementation'...
remote: Enumerating objects: 308, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 308 (delta 23), reused 23 (delta 8), pack-reused 260 (from 2)
Receiving objects: 100% (308/308), 465.96 MiB | 33.25 MiB/s, done.
Resolving deltas: 100% (142/142), done.
Updating files: 100% (47/47), done.


In [2]:
!git clone https://github.com/facebookresearch/dinov3

Cloning into 'dinov3'...
remote: Enumerating objects: 660, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 660 (delta 8), reused 2 (delta 2), pack-reused 625 (from 3)
Receiving objects: 100% (660/660), 13.10 MiB | 29.17 MiB/s, done.
Resolving deltas: 100% (215/215), done.


# Extracting Dino-V3 embeddings for ImageNet-1k

In [3]:
!pip install torchmetrics einops

In [4]:
!pip install datasets

In [5]:
import os
import sys
import pathlib
import urllib.request
import zipfile
import shutil
import numpy as np
import torch
import einops
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2
from PIL import Image
from tqdm import tqdm

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

POS_MEAN_PATH = "/kaggle/working/BSF_implementation/pos_mean.npy"
DINO_REPO = "/kaggle/working/dinov3"
WEIGHTS = "/kaggle/input/models/medicalradhika/dino-v3-model/pytorch/default/1/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"
DATA_ROOT = "/kaggle/working/"
CLASS_IDX = None        # None -> all classes (0-199 for Tiny ImageNet)
# MAX_IMAGES = 500 # Removed to process all images (or all for selected class)
BATCH_SIZE = 64

POS_MEAN = np.load(POS_MEAN_PATH).astype(np.float32)

if DINO_REPO not in sys.path:
    sys.path.insert(0, DINO_REPO)

# ------------------------------------------------------------------
# Download Tiny ImageNet (only once)
# ------------------------------------------------------------------

url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
zip_path = os.path.join(DATA_ROOT, "tiny-imagenet-200.zip")
extract_path = os.path.join(DATA_ROOT, "tiny-imagenet-200")

if not os.path.exists(extract_path):
    print("Downloading Tiny ImageNet...")
    urllib.request.urlretrieve(url, zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_ROOT)
print("Tiny ImageNet ready.")

# ------------------------------------------------------------------
# Delete class folders
# ------------------------------------------------------------------
classes_sorted = sorted(os.listdir(os.path.join(extract_path, "train")))[4:]

for class_subfldr in classes_sorted:
    shutil.rmtree(os.path.join(extract_path, "train", class_subfldr))

# ------------------------------------------------------------------
# Setup directories for storing activations
# ------------------------------------------------------------------
ACTIVATIONS_ROOT = os.path.join(DATA_ROOT, "dino_activations_saved") # Changed name to avoid conflict if already existing
TRAIN_ACTIVATIONS_DIR = os.path.join(ACTIVATIONS_ROOT, "train")
TEST_ACTIVATIONS_DIR = os.path.join(ACTIVATIONS_ROOT, "test")
VAL_ACTIVATIONS_DIR = os.path.join(ACTIVATIONS_ROOT, "val")

os.makedirs(TRAIN_ACTIVATIONS_DIR, exist_ok=True)
os.makedirs(TEST_ACTIVATIONS_DIR, exist_ok=True)
os.makedirs(VAL_ACTIVATIONS_DIR, exist_ok=True)

# ------------------------------------------------------------------
# Load dataset - Collect image paths and labels
# ------------------------------------------------------------------

train_dataset_original = ImageFolder(os.path.join(extract_path, "train"))
test_dataset_original = ImageFolder(os.path.join(extract_path, "test"))

train_image_data = [] # List of (path, label)
for path, label in train_dataset_original.samples:
    if CLASS_IDX is not None and label != CLASS_IDX:
        continue
    train_image_data.append((path, label))
print(f"Collected {len(train_image_data)} train image paths.")

test_image_data = [] # List of (path, label)
for path, label in test_dataset_original.samples:
    if CLASS_IDX is not None and label != CLASS_IDX:
        continue
    test_image_data.append((path, label))
print(f"Collected {len(test_image_data)} test image paths.")

# ------------------------------------------------------------------
# Handle validation data with annotations
# ------------------------------------------------------------------
val_annotations_path = os.path.join(extract_path, "val", "val_annotations.txt")
val_filename_to_synset = {}
with open(val_annotations_path, 'r') as f:
    for line in f.readlines():
        parts = line.strip().split('\t')
        img_filename = parts[0] # e.g., 'val_0.JPEG'
        synset_id = parts[1] # e.g., 'n02124075'
        val_filename_to_synset[img_filename] = synset_id

# Use the class_to_idx mapping from the *training* set for consistency
class_to_idx_mapping = train_dataset_original.class_to_idx

val_image_data = [] # List of (path, label)
val_images_dir = os.path.join(extract_path, "val", "images")
for img_filename in os.listdir(val_images_dir):
    if img_filename in val_filename_to_synset:
        synset_id = val_filename_to_synset[img_filename]
        # Map synset_id to integer label using the training set's mapping
        if synset_id in class_to_idx_mapping:
            class_label = class_to_idx_mapping[synset_id]
            if CLASS_IDX is not None and class_label != CLASS_IDX:
                continue
            full_path = os.path.join(val_images_dir, img_filename)
            val_image_data.append((full_path, class_label))
print(f"Collected {len(val_image_data)} val image paths with correct labels.")


# ------------------------------------------------------------------
# DINO activations - Refactored to save to disk
# ------------------------------------------------------------------

@torch.no_grad()
def save_dino_activations_to_disk(
    image_data, # List of (image_path, image_label) tuples
    output_dir,
    device=None,
    batch_size=64,
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    model = torch.hub.load(
        DINO_REPO,
        "dinov3_vitb16",
        source="local",
        pretrained=False,
    )

    state_dict = torch.load(WEIGHTS, map_location="cpu")
    if "model" in state_dict:
        state_dict = state_dict["model"]
    elif "teacher" in state_dict:
        state_dict = state_dict["teacher"]
    model.load_state_dict(state_dict, strict=True)
    model = model.to(device).eval()

    transform = v2.Compose([
        v2.Resize((224, 224)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])

    saved_activation_records = [] # List of (activation_filepath, image_label)

    for i in tqdm(range(0, len(image_data), batch_size), desc=f"Saving activations to {output_dir}"):
        batch_slice = image_data[i:i + batch_size]
        batch_paths = [item[0] for item in batch_slice]
        batch_labels = [item[1] for item in batch_slice]

        batch_images = [Image.open(p).convert("RGB") for p in batch_paths]
        if not batch_images:
            continue

        batch_tensor = torch.stack([transform(im) for im in batch_images]).to(device)
        feats = model.forward_features(batch_tensor)
        patch_tokens = feats["x_norm_patchtokens"].cpu().numpy() # (batch_size, num_patches, dim)

        for j in range(len(batch_paths)):
            filename = f"activations_{i+j}.npy" # Unique filename for each image's activations
            filepath = os.path.join(output_dir, filename)
            np.save(filepath, patch_tokens[j]) # Save activations for a single image
            saved_activation_records.append((filepath, batch_labels[j]))

    return saved_activation_records

# ------------------------------------------------------------------
# Extract and save activations for each split
# ------------------------------------------------------------------

print("Saving train activations...")
train_activation_records = save_dino_activations_to_disk(
    train_image_data,
    TRAIN_ACTIVATIONS_DIR,
    batch_size=BATCH_SIZE,
)

# print("Saving test activations...")
# test_activation_records = save_dino_activations_to_disk(
#     test_image_data,
#     TEST_ACTIVATIONS_DIR,
#     batch_size=BATCH_SIZE,
# )

print("Saving val activations...")
val_activation_records = save_dino_activations_to_disk(
    val_image_data,
    VAL_ACTIVATIONS_DIR,
    batch_size=BATCH_SIZE,
)

# Print summary of Saved Activations
print("\n--- Summary of Saved Activations ---")
print(f"Train activations saved: {len(train_activation_records)} files")
# print(f"Test activations saved: {len(test_activation_records)} files")
print(f"Val activations saved: {len(val_activation_records)} files")

Extracting...
Tiny ImageNet ready.
Collected 2000 train image paths.
Collected 10000 test image paths.
Collected 200 val image paths with correct labels.
Saving train activations...


Saving activations to /kaggle/working/dino_activations_saved/train: 100%|██████████| 32/32 [00:28<00:00,  1.11it/s]


Saving val activations...


Saving activations to /kaggle/working/dino_activations_saved/val: 100%|██████████| 4/4 [00:02<00:00,  1.49it/s]


--- Summary of Saved Activations ---
Train activations saved: 2000 files
Val activations saved: 200 files


In [6]:
from torch.utils.data import Dataset

class DiskPatchDataset(Dataset):
    def __init__(self, activation_records, pos_mean):
        self.activation_records = activation_records
        self.pos_mean = pos_mean
        self.grid = None # Will be set once first item is loaded

    def __len__(self):
        return len(self.activation_records)

    def __getitem__(self, idx):
        filepath, label = self.activation_records[idx]

        # Load activations from disk for this image
        acts = np.load(filepath).astype(np.float32)

        # Apply the same preprocessing as BSF
        acts = acts - self.pos_mean
        x = einops.rearrange(acts, "p d -> p d")
        x = (
            x
            / np.sqrt((x ** 2).sum(1).mean())
            * np.sqrt(x.shape[1])
        )

        # If grid is not set yet, calculate it from the first loaded item
        if self.grid is None:
            self.grid = int(np.sqrt(acts.shape[0])) # acts.shape[0] is num_patches

        # patch_labels are not needed when loading for the model input, as they are used for evaluation later.
        # We only need the activation `x` for the `PatchDataset` now.

        return torch.from_numpy(x).float()

# Create instances of the new dataset for each split
train_dataset_disk = DiskPatchDataset(train_activation_records, POS_MEAN)
# test_dataset_disk = DiskPatchDataset(test_activation_records, POS_MEAN)
val_dataset_disk = DiskPatchDataset(val_activation_records, POS_MEAN)

print(f"Train DiskPatchDataset created with {len(train_dataset_disk)} samples.")
# print(f"Test DiskPatchDataset created with {len(test_dataset_disk)} samples.")
print(f"Val DiskPatchDataset created with {len(val_dataset_disk)} samples.")

Train DiskPatchDataset created with 2000 samples.
Val DiskPatchDataset created with 200 samples.


In [7]:
# Assuming 224x224 input images and ViT-B/16 model, there are 14x14 patches.
PATCHES_PER_IMAGE = 14 * 14

# Create patch labels for train set
train_image_labels = [record[1] for record in train_activation_records]
train_patch_labels = np.repeat(train_image_labels, PATCHES_PER_IMAGE)

# Create patch labels for test set
# test_image_labels = [record[1] for record in test_activation_records]
# test_patch_labels = np.repeat(test_image_labels, PATCHES_PER_IMAGE)

# Create patch labels for validation set
val_image_labels = [record[1] for record in val_activation_records]
val_patch_labels = np.repeat(val_image_labels, PATCHES_PER_IMAGE)

print(f"Train patch labels shape: {train_patch_labels.shape}")
# print(f"Test patch labels shape: {test_patch_labels.shape}")
print(f"Val patch labels shape: {val_patch_labels.shape}")

Train patch labels shape: (392000,)
Val patch labels shape: (39200,)


In [8]:
import pandas as pd

# Save train activation records to CSV
train_df = pd.DataFrame(train_activation_records, columns=['filepath', 'class_id'])
train_csv_path = os.path.join(TRAIN_ACTIVATIONS_DIR, 'train_activations.csv')
train_df.to_csv(train_csv_path, index=False)
print(f"Train activation mapping saved to: {train_csv_path}")

# # Save test activation records to CSV
# test_df = pd.DataFrame(test_activation_records, columns=['filepath', 'class_id'])
# test_csv_path = os.path.join(TEST_ACTIVATIONS_DIR, 'test_activations.csv')
# test_df.to_csv(test_csv_path, index=False)
# print(f"Test activation mapping saved to: {test_csv_path}")

# Save validation activation records to CSV
val_df = pd.DataFrame(val_activation_records, columns=['filepath', 'class_id'])
val_csv_path = os.path.join(VAL_ACTIVATIONS_DIR, 'val_activations.csv')
val_df.to_csv(val_csv_path, index=False)
print(f"Validation activation mapping saved to: {val_csv_path}")

Train activation mapping saved to: /kaggle/working/dino_activations_saved/train/train_activations.csv
Validation activation mapping saved to: /kaggle/working/dino_activations_saved/val/val_activations.csv


# Importing Modules

In [9]:
import os
os.chdir("/kaggle/working/BSF_implementation")

import sys
from pathlib import Path

# Add the project root (parent of notebooks) to Python's search path
sys.path.append(str(Path().resolve().parent))

import os
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from models.vanilla_bsf import build_model

# Configuration

In [10]:
# ACTIVATION_FOLDER = "data/rabit_activations_dinov3"
INPUT_DIM = 768
NUM_BLOCKS = 256
BLOCK_SIZE = 3
TOP_K = 8
BATCH_SIZE = 64 # Changed BATCH_SIZE to 512 to ensure train_loader gets batches
LR = 3e-3
EPOCHS = 300
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Dataloader

In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset_disk,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

# test_loader = DataLoader(
#     test_dataset_disk,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     drop_last=False,
# )

val_loader = DataLoader(
    val_dataset_disk,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
)
print(f"Re-created Val DataLoader with {len(val_loader)} batches (batch size {BATCH_SIZE} images).")


print(f"Train DataLoader created with {len(train_loader)} batches (batch size {BATCH_SIZE} images).")
# print(f"Test DataLoader created with {len(test_loader)} batches (batch size {BATCH_SIZE} images).")
print(f"Val DataLoader created with {len(val_loader)} batches (batch size {BATCH_SIZE} images).")

Re-created Val DataLoader with 4 batches (batch size 64 images).
Train DataLoader created with 31 batches (batch size 64 images).
Val DataLoader created with 4 batches (batch size 64 images).


In [12]:
model = build_model(
    input_dim=INPUT_DIM,
    num_blocks=NUM_BLOCKS,
    block_size=BLOCK_SIZE,
    top_k=TOP_K,
)

model.to(DEVICE)

Vanilla_BSF(
  (W): Linear(in_features=768, out_features=768, bias=False)
  (D): Linear(in_features=768, out_features=768, bias=False)
)

# Training Vanilla BSF

In [13]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
)

criterion = nn.MSELoss()

In [14]:
history = []

best_loss = float("inf")

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    dead_block_counter = None

    for x in tqdm(train_loader):

        x = x.to(DEVICE)

        # Flatten x from (batch_size, patches_per_image, input_dim) to (batch_size * patches_per_image, input_dim)
        x_flattened = einops.rearrange(x, 'b p d -> (b p) d')

        output = model(x_flattened)

        reconstruction = output["reconstruction"]

        loss = criterion(
            reconstruction,
            x_flattened,
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        ###################################
        # Dead blocks
        ###################################

        mask = output["active_mask"]

        if dead_block_counter is None:

            dead_block_counter = mask.sum(0)

        else:

            dead_block_counter += mask.sum(0)

    epoch_loss = running_loss / len(train_loader)

    history.append(epoch_loss)

    dead_blocks = (dead_block_counter == 0).sum().item()

    print(
        f"Epoch {epoch+1:03d}"
        f" | Loss = {epoch_loss:.6f}"
        f" | Dead Blocks = {dead_blocks}"
    )

    if best_loss < 0.2:
      break

    if epoch_loss < best_loss:

        best_loss = epoch_loss

        torch.save(
            model.state_dict(),
            "improved_vanilla_bsf_dinov3.pth",
        )

100%|██████████| 31/31 [00:04<00:00,  7.65it/s]


Epoch 001 | Loss = 0.632164 | Dead Blocks = 0


100%|██████████| 31/31 [00:03<00:00,  8.11it/s]


Epoch 002 | Loss = 0.458241 | Dead Blocks = 0


100%|██████████| 31/31 [00:03<00:00,  8.33it/s]


...

Epoch 298 | Loss = 0.307115 | Dead Blocks = 0


100%|██████████| 31/31 [00:03<00:00,  8.51it/s]


Epoch 299 | Loss = 0.305858 | Dead Blocks = 0


100%|██████████| 31/31 [00:03<00:00,  8.41it/s]

Epoch 300 | Loss = 0.305354 | Dead Blocks = 0


# Single Concept F1 score calculation

In [15]:
import numpy as np
import torch
import einops # Import einops for reshaping

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from tqdm import tqdm # Import tqdm for progress bar


##########################################################################
# Evaluate ONE block (modified to handle data_loader directly)
##########################################################################

@torch.no_grad()
def evaluate_block(
    model,
    data_loader,
    patch_labels, # These are all patch labels for the entire dataset split
    block_idx,
    class_id,
):
    device = next(model.parameters()).device
    model.eval() # Set model to evaluation mode

    X_features_list = []
    y_labels_list = []
    total_patches_processed = 0 # To correctly slice patch_labels

    # Assuming PATCHES_PER_IMAGE is defined globally or can be derived
    # from the model or data_loader structure.
    # For instance, if input images are 224x224 and patch size is 16x16, then 14x14 = 196 patches
    # PATCHES_PER_IMAGE = 14 * 14 # This should be available from earlier cells

    for x_batch in tqdm(data_loader, desc=f"Extracting features for block {block_idx}, class {class_id}"):
        x_batch = x_batch.to(device)

        # Get actual batch size (number of images in the current batch)
        batch_size_actual = x_batch.shape[0]

        # Flatten x from (batch_size, patches_per_image, input_dim) to (batch_size * patches_per_image, input_dim)
        x_flattened = einops.rearrange(x_batch, 'b p d -> (b p) d')

        out = model(x_flattened)

        # Extract features for the current block
        z_blocks_batch = out["z_blocks"][:, block_idx, :] # Shape: (batch_size * patches_per_image, block_size)
        X_features_list.append(z_blocks_batch.cpu().numpy())

        # Determine the corresponding patch labels for this batch
        num_patches_in_batch = batch_size_actual * PATCHES_PER_IMAGE
        current_patch_labels_batch = patch_labels[
            total_patches_processed : total_patches_processed + num_patches_in_batch
        ]
        y_labels_list.append(current_patch_labels_batch)

        total_patches_processed += num_patches_in_batch

    # Concatenate all features and labels collected from the data_loader
    X = np.concatenate(X_features_list, axis=0)
    y = np.concatenate(y_labels_list, axis=0)

    # Binary labels for the current class_id
    y_binary = (y == class_id).astype(np.int32)

    ####################################################
    # Linear probe
    ####################################################

    clf = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        solver="lbfgs",
        n_jobs=-1, # Use all available CPU cores for Logistic Regression
    )

    clf.fit(
        X,
        y_binary,
    )

    ####################################################
    # Projection scores
    ####################################################

    scores = clf.decision_function(X)

    ####################################################
    # Threshold sweep
    ####################################################

    precision, recall, thresholds = precision_recall_curve(
        y_binary,
        scores,
    )

    f1 = (
        2
        * precision[:-1]
        * recall[:-1]
        /
        (
            precision[:-1]
            + recall[:-1]
            + 1e-12
        )
    )

    best = np.argmax(f1)

    return {
        "block": block_idx,
        "classifier": clf,
        "best_f1": float(f1[best]),
        "best_threshold": float(thresholds[best]),
        "precision": float(precision[best]),
        "recall": float(recall[best]),
        "scores": scores,
    }


##########################################################################
# Evaluate ALL blocks (modified)
##########################################################################

def evaluate_all_blocks(
    model,
    data_loader,
    patch_labels,
    class_id,
):
    # Determine num_blocks from the model
    num_blocks = NUM_BLOCKS # Access num_blocks from the model instance

    results = []

    for block_idx in range(num_blocks): # Loop over each block
        result = evaluate_block(
            model,
            data_loader,
            patch_labels,
            block_idx,
            class_id,
        )
        results.append(result)

    ####################################################
    # Best concept block
    ####################################################

    winner = max(
        results,
        key=lambda x: x["best_f1"],
    )

    return winner, results


##########################################################################
# Complete pipeline (modified)
##########################################################################

def evaluate_class(
    model,
    data_loader,
    patch_labels,
    class_id,
):
    # The z_blocks extraction is now handled inside evaluate_block
    # No need for extract_block_codes here.

    # `extract_block_codes` is no longer needed/used in this flow, as its logic is now within `evaluate_block`
    # I'm removing the original `extract_block_codes` from the snippet to avoid confusion if it were left in.

    winner, all_results = evaluate_all_blocks(
        model,
        data_loader,
        patch_labels,
        class_id,
    )

    return {
        "winner_block": winner["block"],
        "best_f1": winner["best_f1"],
        "best_threshold": winner["best_threshold"],
        "precision": winner["precision"],
        "recall": winner["recall"],
        "classifier": winner["classifier"],
        "scores": winner["scores"],
        "all_results": all_results,
    }

##########################################################################
# Evaluate multiple classes
##########################################################################

def evaluate_classes(
    model,
    data_loader,
    patch_labels,
    class_ids,
    verbose=True,
):
    """
    Evaluates every class independently.

    For every class:
        - trains logistic regression for every block
        - finds the best threshold
        - selects the best block
        - stores precision, recall and F1

    Parameters
    ----------
    class_ids : list[int]
        List of semantic class ids.

    Returns
    -------
    dict containing:
        average_f1
        average_precision
        average_recall
        class_results
    """

    class_results = {}

    f1_scores = []
    precision_scores = []
    recall_scores = []

    iterator = tqdm(class_ids, desc="Evaluating classes") if verbose else class_ids

    for class_id in iterator:

        result = evaluate_class(
            model=model,
            data_loader=data_loader,
            patch_labels=patch_labels,
            class_id=class_id,
        )

        class_results[class_id] = result

        f1_scores.append(result["best_f1"])
        precision_scores.append(result["precision"])
        recall_scores.append(result["recall"])

    return {
        "average_f1": float(np.mean(f1_scores)),
        "average_precision": float(np.mean(precision_scores)),
        "average_recall": float(np.mean(recall_scores)),
        "class_results": class_results,
    }

In [16]:
class_ids = [0, 1, 2, 3]

results = evaluate_classes(
    model=model,
    data_loader=val_loader,
    patch_labels=val_patch_labels,
    class_ids=class_ids,
)

print("Average F1       :", results["average_f1"])
print("Average Precision:", results["average_precision"])
print("Average Recall   :", results["average_recall"])

for cls, res in results["class_results"].items():
    print(
        f"Class {cls:2d} | "
        f"Block {res['winner_block']:2d} | "
        f"F1={res['best_f1']:.4f} | "
        f"P={res['precision']:.4f} | "
        f"R={res['recall']:.4f}"
    )

Extracting features for block 0, class 0: 100%|██████████| 4/4 [00:00<00:00, 12.85it/s]

Extracting features for block 1, class 0: 100%|██████████| 4/4 [00:00<00:00,  6.51it/s]

Extracting features for block 2, class 0: 100%|██████████| 4/4 [00:00<00:00, 12.40it/s]

Extracting features for block 3, class 0: 100%|██████████| 4/4 [00:00<00:00, 12.78it/s]

Extracting features for block 4, class 0: 100%|██████████| 4/4 [00:00<00:00, 11.29it/s]

Extracting features for block 5, class 0: 100%|██████████| 4/4 [00:00<00:00, 10.58it/s]

Extracting features for block 6, class 0: 100%|██████████| 4/4 [00:00<00:00, 11.00it/s]

Extracting features for block 7, class 0: 100%|██████████| 4/4 [00:00<00:00, 11.08it/s]

Extracting features for block 8, class 0: 100%|██████████| 4/4 [00:00<00:00, 10.96it/s]

Extracting features for block 9, class 0: 100%|██████████| 4/4 [00:00<00:00, 11.90it/s]

Extracting features for block 10, class 0: 100%|██████████| 4/4 [00:00<00:00, 11.64it/s]

Extracting features 

Average F1       : 0.4860909039542919
Average Precision: 0.5784077167141789
Average Recall   : 0.6840561224489796
Class  0 | Block 195 | F1=0.4971 | P=0.9140 | R=0.3414
Class  1 | Block  2 | F1=0.5774 | P=0.8403 | R=0.4398
Class  2 | Block  2 | F1=0.4366 | P=0.2802 | R=0.9876
Class  3 | Block  2 | F1=0.4332 | P=0.2791 | R=0.9674
